## GenAI Disclosure Statement

In this course, generative AI tools were used as learning aids. All analysis and
conclusions are the original work of the project team.

---

# Notebook 02 — Baseline Model Development
### DNSC 6330 Responsible Machine Learning Capstone | GWU

**Purpose:** Train logistic regression (interpretable baseline) and XGBoost GBM (candidate deployment model).
Select operating threshold. Produce performance metrics for audit record.

**Inputs:** `data/processed/train.parquet`, `val.parquet`, `test.parquet`  
**Outputs:** `models/*.pkl`, `tables/metrics_table.csv`, `figures/roc_curve.png`, `figures/pr_curve.png`


In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")

import src.models as models
importlib.reload(models)

from src.models import (
    train_logistic_regression, train_gbm, evaluate_model,
    save_model, select_threshold
)

SEED = 42
np.random.seed(SEED)

PROC_DIR = os.path.join(os.getcwd(), "..", "data", "processed")
MODELS_DIR = os.path.join(os.getcwd(), "..", "models")
TABLES_DIR = os.path.join(os.getcwd(), "..", "tables")
FIG_DIR = os.path.join(os.getcwd(), "..", "figures")

for d in [MODELS_DIR, TABLES_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)


In [2]:
# Load processed splits
train_df = pd.read_parquet(os.path.join(PROC_DIR, "train.parquet"))
val_df = pd.read_parquet(os.path.join(PROC_DIR, "val.parquet"))
test_df = pd.read_parquet(os.path.join(PROC_DIR, "test.parquet"))

# Keep the modeling feature set aligned with the processed export
NON_FEATURE_COLS = [
    "y",
    "state_code",
    "derived_race",
    "derived_sex",
    "derived_ethnicity",
    "applicant_sex",
    "applicant_age",
    "race_sex_intersection",
]

feature_path = os.path.join(PROC_DIR, "feature_columns.txt")
with open(feature_path, "r") as f:
    feature_cols = [line.strip() for line in f if line.strip()]
feature_cols = [c for c in feature_cols if c in train_df.columns and c not in NON_FEATURE_COLS]

X_train = train_df[feature_cols].copy()
y_train = train_df["y"].values
X_val = val_df[feature_cols].copy()
y_val = val_df["y"].values
X_test = test_df[feature_cols].copy()
y_test = test_df["y"].values

numeric_feature_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_feature_cols = [c for c in X_train.columns if c not in numeric_feature_cols]

print(f"Training features: {len(feature_cols)}")
print(f"Numeric features:   {len(numeric_feature_cols)}")
print(f"Categorical feats:   {len(categorical_feature_cols)}")
print(f"Train:  {len(X_train):,} rows | positive rate: {y_train.mean():.3f}")
print(f"Val:    {len(X_val):,} rows | positive rate: {y_val.mean():.3f}")
print(f"Test:   {len(X_test):,} rows | positive rate: {y_test.mean():.3f}")


Training features: 30
Numeric features:   30
Categorical feats:   0
Train:  5,553,702 rows | positive rate: 0.758
Val:    1,190,079 rows | positive rate: 0.758
Test:   1,190,080 rows | positive rate: 0.758


## Section 2.2 — Logistic Regression (Interpretable Baseline)

The logistic regression serves as the interpretable baseline. Its coefficients provide
direct evidence about feature-direction relationships. Any GBM that is proposed for
deployment should meaningfully outperform this baseline to justify the explainability cost.

**Hyperparameter choices:** C=0.1 (regularization), balanced class weights, lbfgs solver.


In [3]:
print("Training Logistic Regression...")
lr_model = train_logistic_regression(X_train, y_train, C=0.1, random_state=SEED)

lr_val_metrics  = evaluate_model(lr_model, X_val,  y_val,  model_name="LogReg", split_name="Val")
lr_test_metrics = evaluate_model(lr_model, X_test, y_test, model_name="LogReg", split_name="Test")

lr_model_path = save_model(lr_model, "logreg", models_dir=MODELS_DIR)


Training Logistic Regression...

  LogReg — Val Set Metrics
  AUC:       0.7440
  PR-AUC:    0.8851
  KS:        0.3872
  Brier:     0.2069
  Accuracy:  0.6652  (threshold=0.5)
  Precision: 0.8879
  Recall:    0.6388
  F1:        0.7430
  CM: TP=575972 FP=72711 FN=325669 TN=215727

  LogReg — Test Set Metrics
  AUC:       0.7442
  PR-AUC:    0.8856
  KS:        0.3878
  Brier:     0.2068
  Accuracy:  0.6655  (threshold=0.5)
  Precision: 0.8881
  Recall:    0.6390
  F1:        0.7433
  CM: TP=576181 FP=72591 FN=325461 TN=215847
[save] Model saved: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/notebooks/../models/logreg_v20260506.pkl
[save] Meta saved:  /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/notebooks/../models/logreg_v20260506_meta.json


## Section 2.3 — Gradient-Boosted Model (Candidate Deployment Model)

XGBoost with early stopping on the validation set.
Hyperparameters are tuned conservatively to avoid overfitting to the training distribution.

**Key choices:**
- `scale_pos_weight` set to class imbalance ratio to handle label imbalance
- `max_depth=4`: limits complexity; improves generalization and reduces proxy encoding
- `early_stopping_rounds=30`: prevents overfitting


In [4]:
print("Training Gradient-Boosted Model (XGBoost)...")

# The shared helper handles preprocessing, fitting, and raw-data inference.
gbm_model = train_gbm(X_train, y_train, X_val, y_val, random_state=SEED)

# Validation and test metrics
#print("Evaluating GBM...")
gbm_val_metrics = evaluate_model(gbm_model, X_val, y_val, model_name="GBM", split_name="Val")
gbm_test_metrics = evaluate_model(gbm_model, X_test, y_test, model_name="GBM", split_name="Test")

gbm_model_path = save_model(gbm_model, "gbm", models_dir=MODELS_DIR)


Training Gradient-Boosted Model (XGBoost)...
[GBM] Validation AUC: 0.8119

  GBM — Val Set Metrics
  AUC:       0.8119
  PR-AUC:    0.9186
  KS:        0.4707
  Brier:     0.1740
  Accuracy:  0.7386  (threshold=0.5)
  Precision: 0.8951
  Recall:    0.7419
  F1:        0.8114
  CM: TP=668969 FP=78388 FN=232672 TN=210050

  GBM — Test Set Metrics
  AUC:       0.8127
  PR-AUC:    0.9189
  KS:        0.4722
  Brier:     0.1738
  Accuracy:  0.7389  (threshold=0.5)
  Precision: 0.8958
  Recall:    0.7417
  F1:        0.8115
  CM: TP=668715 FP=77788 FN=232927 TN=210650
[save] Model saved: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/notebooks/../models/gbm_v20260506.pkl
[save] Meta saved:  /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/notebooks/../models/gbm_v20260506_meta.json


## Section 2.4 — Metrics Comparison Table

Side-by-side comparison of both models. The GBM should outperform LR; if it does not,
the LR baseline should be preferred for its transparency.


In [5]:
all_metrics = [lr_val_metrics, lr_test_metrics, gbm_val_metrics, gbm_test_metrics]
metrics_df = pd.DataFrame(all_metrics)
display_cols = ["model", "split", "threshold", "n", "auc", "pr_auc", "ks", "brier",
                "accuracy", "precision", "recall", "f1", "tp", "fp", "fn", "tn"]
metrics_df = metrics_df[[c for c in display_cols if c in metrics_df.columns]]
display(metrics_df)
metrics_df.to_csv(os.path.join(TABLES_DIR, "metrics_table_default_threshold.csv"), index=False)
print("Metrics table saved.")


,model,split,threshold,n,auc,pr_auc,ks,brier,accuracy,precision,recall,f1,tp,fp,fn,tn
0,LogReg,Val,0.5,1190079,0.7440,0.8851,0.3872,0.2069,0.6652,0.8879,0.6388,0.7430,575972,72711,325669,215727
1,LogReg,Test,0.5,1190080,0.7442,0.8856,0.3878,0.2068,0.6655,0.8881,0.6390,0.7433,576181,72591,325461,215847
2,GBM,Val,0.5,1190079,0.8119,0.9186,0.4707,0.1740,0.7386,0.8951,0.7419,0.8114,668969,78388,232672,210050
3,GBM,Test,0.5,1190080,0.8127,0.9189,0.4722,0.1738,0.7389,0.8958,0.7417,0.8115,668715,77788,232927,210650


Metrics table saved.


## Section 2.5 — Threshold Selection

The default threshold of 0.5 is arbitrary. We select the operating threshold by maximizing
F1 on the **validation set** (not the test set — to avoid peeking).

After selecting the threshold, we re-evaluate on the test set and examine the fairness
impact of the chosen threshold.

**Decision D-007:** F1 maximization on validation set. This decision will be revisited
in Notebook 04 after fairness analysis.


In [6]:
# Get validation probabilities
gbm_val_probs = gbm_model.predict_proba(X_val)[:, 1]
lr_val_probs  = lr_model.predict_proba(X_val)[:, 1]

# Select threshold
gbm_threshold = select_threshold(y_val, gbm_val_probs, strategy="f1")
lr_threshold  = select_threshold(y_val, lr_val_probs,  strategy="f1")
print(f"GBM operating threshold (F1-optimal on val): {gbm_threshold}")
print(f"LR  operating threshold (F1-optimal on val): {lr_threshold}")

# Re-evaluate both models at selected thresholds on test set
gbm_test_final = evaluate_model(gbm_model, X_test, y_test,
                                threshold=gbm_threshold,
                                model_name="GBM (F1-threshold)",
                                split_name="Test")
lr_test_final  = evaluate_model(lr_model,  X_test, y_test,
                                threshold=lr_threshold,
                                model_name="LogReg (F1-threshold)",
                                split_name="Test")

final_metrics = pd.DataFrame([gbm_test_final, lr_test_final])
final_metrics.to_csv(os.path.join(TABLES_DIR, "metrics_table_final.csv"), index=False)
display(final_metrics)
print(f"\nSelected operating threshold for GBM: {gbm_threshold}")


GBM operating threshold (F1-optimal on val): 0.2575
LR  operating threshold (F1-optimal on val): 0.2476

  GBM (F1-threshold) — Test Set Metrics
  AUC:       0.8127
  PR-AUC:    0.9189
  KS:        0.4722
  Brier:     0.1738
  Accuracy:  0.8136  (threshold=0.2575)
  Precision: 0.8253
  Recall:    0.9565
  F1:        0.8861
  CM: TP=862418 FP=182569 FN=39224 TN=105869

  LogReg (F1-threshold) — Test Set Metrics
  AUC:       0.7442
  PR-AUC:    0.8856
  KS:        0.3878
  Brier:     0.2068
  Accuracy:  0.7709  (threshold=0.2476)
  Precision: 0.7839
  Recall:    0.9632
  F1:        0.8643
  CM: TP=868483 FP=239457 FN=33159 TN=48981


,model,split,threshold,n,auc,pr_auc,brier,ks,accuracy,precision,recall,f1,tp,fp,fn,tn,positive_rate_predicted,positive_rate_actual
0,GBM (F1-threshold),Test,0.2575,1190080,0.8127,0.9189,0.1738,0.4722,0.8136,0.8253,0.9565,0.8861,862418,182569,39224,105869,0.8781,0.7576
1,LogReg (F1-threshold),Test,0.2476,1190080,0.7442,0.8856,0.2068,0.3878,0.7709,0.7839,0.9632,0.8643,868483,239457,33159,48981,0.9310,0.7576



Selected operating threshold for GBM: 0.2575


## Section 2.6 — ROC and PR Curves

Standard performance visualization. Both models plotted together for comparison.


In [7]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# ROC curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for model, probs, label, color in [
    (gbm_model, gbm_model.predict_proba(X_test)[:, 1], "GBM", "steelblue"),
    (lr_model,  lr_model.predict_proba(X_test)[:, 1],  "Logistic Regression", "coral"),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f"{label} (AUC={roc_auc:.4f})", color=color, lw=2)

axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve — Test Set")
axes[0].legend()
axes[0].grid(alpha=0.3)

# PR curves
for model, probs, label, color in [
    (gbm_model, gbm_model.predict_proba(X_test)[:, 1], "GBM", "steelblue"),
    (lr_model,  lr_model.predict_proba(X_test)[:, 1],  "Logistic Regression", "coral"),
]:
    precision, recall, _ = precision_recall_curve(y_test, probs)
    pr_auc_val = auc(recall, precision)
    axes[1].plot(recall, precision, label=f"{label} (PR-AUC={pr_auc_val:.4f})", color=color, lw=2)

axes[1].axhline(y=y_test.mean(), color="gray", linestyle="--", lw=1, label="Baseline (prevalence)")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve — Test Set")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
roc_path = os.path.join(FIG_DIR, "roc_pr_curves.png")
plt.savefig(roc_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {roc_path}")


Saved: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/notebooks/../figures/roc_pr_curves.png


## Section 2.7 — Confusion Matrix at Operating Threshold

In [8]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

gbm_test_probs = gbm_model.predict_proba(X_test)[:, 1]
gbm_test_preds = (gbm_test_probs >= gbm_threshold).astype(int)

cm = confusion_matrix(y_test, gbm_test_preds, labels=[0, 1])
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=["Denied (0)", "Approved (1)"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"GBM Confusion Matrix\n(Test Set, threshold={gbm_threshold})", fontsize=12)
plt.tight_layout()
cm_path = os.path.join(FIG_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {cm_path}")
print(f"\nTP={cm[1,1]:,}  FP={cm[0,1]:,}")
print(f"FN={cm[1,0]:,}  TN={cm[0,0]:,}")
print(f"FNR = {cm[1,0] / (cm[1,0] + cm[1,1]):.4f}")
print(f"FPR = {cm[0,1] / (cm[0,1] + cm[0,0]):.4f}")
print("\n Notebook 02 complete.")


Saved: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/notebooks/../figures/confusion_matrix.png

TP=862,418  FP=182,569
FN=39,224  TN=105,869
FNR = 0.0435
FPR = 0.6330

 Notebook 02 complete.
